# Лабораторная работа 6. Вариант 1
**Тема.** Решение NP-полных задач на Flow-shop

## **Задача 1.**
$$
F_3 | \space | C_{max}
$$ 
Данная задача, в отличии от $F_{2}$, является NP-трудной, то есть простой метод Джонсона тут не подойдет.

In [1]:
tasks = {
    'J1': (1, 13, 6),
    'J2': (10, 12, 18),
    'J3': (17, 9, 13),
    'J4': (12, 17, 2),
    'J5': (11, 3, 5)
}

### Алгоритм
Оптимальным решением здесь будет метод ветвей и границ [1, 4.7]. Тут вместо того чтобы проверять все $n!$ вариантов перестановок работ, строится дерево решений и отсекаются целые ветви, если в них точно нет результата лучше, чем уже найденный. То есть:
- строим дерево, где на каждом уровне выбирается следующая работа в очередь;
- для каждого узла считается нижнюю оценку — это минимально возможное время $C_{max}$, которое получается, если продолжим строить расписание из этого узла.

Чтобы отсечь ветку, необходимо доказать, что лучшее $C_{max}$ не будет оптимальным. Для этого считаются нижние оценки (три, так как три станка):
</br></br>
$$
LB_{1} = t_{1} + \sum_{j \in rest} p_{j1}+ min_{j \in rest}(p_{j2} + p_{j3})
$$
</br>
$$
LB_{2} = t_{2} + \sum_{j \in rest} p_{j2}+ min_{j \in rest}(p_{j3})
$$
</br>
$$
LB_{3} = t_{3} + \sum_{j \in rest} p_{j3}
$$
где:
- $t_n$ -- текущее время на $M_n$
- rest -- остаток времени

В итоге берем $LB=max(LB_1,LB_2,LB_3)$

In [2]:
def get_lower_bound(partial_schedule, remaining_ids, jobs_data):
    if not remaining_ids:
        c1, c2, c3 = calculate_times(partial_schedule, jobs_data)
        return c3
    
    c1, c2, c3 = calculate_times(partial_schedule, jobs_data)
    
    sum_p1 = sum(jobs_data[j][0] for j in remaining_ids)
    min_p23 = min(jobs_data[j][1] + jobs_data[j][2] for j in remaining_ids)
    lb1 = c1 + sum_p1 + min_p23
    
    sum_p2 = sum(jobs_data[j][1] for j in remaining_ids)
    min_p3 = min(jobs_data[j][2] for j in remaining_ids)
    lb2 = c2 + sum_p2 + min_p3
    
    sum_p3 = sum(jobs_data[j][2] for j in remaining_ids)
    lb3 = c3 + sum_p3
    
    return max(lb1, lb2, lb3)

Имея алгоритм отсечения мы можем просто обойти в глубину дерево возможных решений, выбрасывая ветки с высоким $LB$.

In [3]:
def branch_and_bound(current_schedule, remaining_ids, jobs_data):
    global best_cmax, best_order
    
    lb = get_lower_bound(current_schedule, remaining_ids, jobs_data)
    
    if lb >= best_cmax:
        return

    if not remaining_ids:
        _, _, final_c3 = calculate_times(current_schedule, jobs_data)
        if final_c3 < best_cmax:
            best_cmax = final_c3
            best_order = current_schedule
        return

    for i in range(len(remaining_ids)):
        next_job = remaining_ids[i]
        new_remaining = remaining_ids[:i] + remaining_ids[i+1:]
        branch_and_bound(current_schedule + [next_job], new_remaining, jobs_data)

Введем функцию подсчета времени

In [4]:
def calculate_times(partial_schedule, jobs_data):
    c1, c2, c3 = 0, 0, 0
    for j_id in partial_schedule:
        p1, p2, p3 = jobs_data[j_id]
        c1 += p1
        c2 = max(c1, c2) + p2
        c3 = max(c2, c3) + p3
    return c1, c2, c3

Проведем моделирование и выведем результаты

In [5]:
best_cmax = float('inf')
best_order = []
branch_and_bound([], list(tasks.keys()), tasks)

print(f"Точное оптимальное расписание: {best_order}")
print(f"Минимальный Makespan (C_max): {best_cmax}")

Точное оптимальное расписание: ['J1', 'J2', 'J3', 'J4', 'J5']
Минимальный Makespan (C_max): 65


### Алгоритм 
Для решения таких задач существуют и эвристики, например Campbell-Dudek-Smith [1, 4.6.2]. Это по сути расширение правила Джонсона.

In [6]:
def solve_johnson_rule(jobs):
    left = []
    right = []
    remaining_jobs = list(jobs)
    
    while remaining_jobs:
        min_val = float('inf')
        best_job = None
        machine = None
        
        for job in remaining_jobs:
            if job['p1'] < min_val:
                min_val = job['p1']
                best_job = job
                machine = 1
            if job['p2'] < min_val:
                min_val = job['p2']
                best_job = job
                machine = 2
        
        if machine == 1:
            left.append(best_job['id'])
        else:
            right.insert(0, best_job['id'])
            
        remaining_jobs.remove(best_job)
        
    optimal_order = left + right
    return optimal_order

Правило Джонсона хорошо работает для двух станков, в связи с этим основной идеей эвристики является свренуть $n$ станков в два виртуальных. Идею можно сформулировать так:
> Если у нас есть $m$ станков, мы можем создать $m−1$ различных вариантов двухстаночных задач. Для каждого варианта мы находим оптимальную последовательность по правилу Джонсона, считаем реальный $C_{max}$ для исходной задачи и выбираем тот порядок, который дал самый короткий график.

Сама свертка происходит по следующему принципу, существуют некоторое число итераций (от $k=1$ до $m−1$), на каждой из которых мы формируем два виртуальных станка ($M_1'$ и $M_2'$):
- время на первом виртуальном станке — это сумма времен на первых $k$ реальных станках:
  $$
  p_{j1}'= \sum_{i=1}^{k}p_{ji}
  $$
- время на втором, соотвественно, сумма времен на последних $k$ станках:
  $$
  p_{j2}'= \sum_{i=1}^{k}p_{j, m-1+1}
  $$

In [7]:
def calculate_makespan_f3(order, p1, p2, p3):
    c1, c2, c3 = 0, 0, 0
    for j in order:
        c1 += p1[j]
        c2 = max(c1, c2) + p2[j]
        c3 = max(c2, c3) + p3[j]
    return c3


Далее применяется правило Джонсона и выбирается лучшая последовательность. Смоделируем процесс.

In [8]:
p1 = {id: v[0] for id, v in tasks.items()}
p2 = {id: v[1] for id, v in tasks.items()}
p3 = {id: v[2] for id, v in tasks.items()}

results = []

j1_data = [{'id': id, 'p1': v[0], 'p2': v[2]} for id, v in tasks.items()]
order1 = solve_johnson_rule(j1_data)
res1 = calculate_makespan_f3(order1, p1, p2, p3)

j2_data = [{'id': id, 'p1': v[0]+v[1], 'p2': v[1]+v[2]} for id, v in tasks.items()]
order2 = solve_johnson_rule(j2_data)
res2 = calculate_makespan_f3(order2, p1, p2, p3)

Выведем результат

In [9]:
if res1 <= res2:
    print(f"Лучшее расписание по CDS: {order1}")
    print(f"C_max (Makespan) = {res1}")
else:
    print(f"Лучшее расписание по CDS: {order2}")
    print(f"C_max (Makespan) = {res2}")

Лучшее расписание по CDS: ['J1', 'J2', 'J3', 'J4', 'J5']
C_max (Makespan) = 65


## References
1. ALGORITHMS FOR SEQUENCING AND SCHEDULING, Ibrahim M. Alharkan